# Person 4 (Neha): EXPLORATORY checks (plan section 8)
**Everything here is exploratory.** Not part of the pass/fail criteria. Plan: `docs/correction_model_plan_v1.md` (frozen).
1. Results by task type and difficulty (no refit; uses the final run's corrected scores).
2. Refit on the third set without the Coding pairs (AI-drafted), then rerun the section 7 criteria.
The final model and its verdict are NOT changed by these checks.
Library and functions are copied unchanged from `Person04d_FinalRun.ipynb`.

## 1. Upload 5 files
`scores_thirdset_v1.csv`, `features_thirdset_v1.csv`, `scores_confirmatory_v1.csv`, `features_confirmatory_v1.csv`, `corrected_scores_conf_v1.csv`

In [ ]:
from google.colab import files
up = files.upload()
print(list(up))

## A. Library (unchanged)

In [ ]:
from IPython.display import display
import json, hashlib, itertools
import numpy as np, pandas as pd
from scipy import stats
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import cohen_kappa_score, roc_auc_score

SEED = 0
FEATS = ['hi_score', 'hi_tokens', 'hi_words', 'hi_fertility']      # allowed inputs (Hinglish only)
CUTS = {'10%': 0.5949, '20%': 0.5094, '30%': 0.4651, '50%': 0.4066, '70%': 0.3372}   # from config_v4.json

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''): h.update(chunk)
    return h.hexdigest()

def compute_features(df, tok, text_col='hinglish'):
    """Same recipe as Person02 / features_confirmatory_v1.csv: words = len(text.split()),
    tokens = len(router tokenizer tokens), fertility = tokens / words. NFC-normalised, stripped text."""
    import unicodedata
    texts = [unicodedata.normalize('NFC', str(x).strip()) for x in df[text_col]]
    f = pd.DataFrame({'pair_id': df['pair_id'].values})
    f[f'{text_col}_words'] = [len(t.split()) for t in texts]
    f[f'{text_col}_tokens'] = [len(tok.tokenize(t)) for t in texts]
    f[f'{text_col}_fertility'] = f[f'{text_col}_tokens'] / f[f'{text_col}_words']
    return f

def check_cuts(config_path):
    """The plan's cut-offs must match config_v4.json to four decimals."""
    c = json.load(open(config_path))
    ref = {'10%': c['threshold_10pct'], '20%': c['threshold_20pct'], '30%': c['threshold_30pct'],
           '50%': c['threshold'], '70%': c['threshold_70pct']}
    for k, v in ref.items():
        assert round(v, 4) == CUTS[k], f'cut-off {k}: config {v} vs plan {CUTS[k]}'
    return ref

# ---------- methods (all fitted on training rows only) ----------
class M0:                      # no correction
    name = 'M0 none'
    def fit(self, X, e): return self
    def predict(self, X): return X['hi_score'].to_numpy()

class M2:                      # quantile mapping, K knots, clipped at the ends
    def __init__(self, K=21): self.K = K; self.name = f'M2 quantile map K={K}'
    def fit(self, X, e):
        q = np.linspace(0, 1, self.K)
        self.hk = np.quantile(X['hi_score'], q); self.ek = np.quantile(e, q); return self
    def predict(self, X): return np.interp(X['hi_score'], self.hk, self.ek)

class M3:                      # linear regression on the allowed inputs
    name = 'M3 linear'
    def fit(self, X, e): self.m = LinearRegression().fit(X[FEATS], e); return self
    def predict(self, X): return np.clip(self.m.predict(X[FEATS]), 0, 1)

class M4:                      # small gradient boosting
    def __init__(self, depth=2, n=50): self.depth, self.n = depth, n; self.name = f'M4 boosting d={depth} n={n}'
    def fit(self, X, e):
        self.m = GradientBoostingRegressor(max_depth=self.depth, n_estimators=self.n, learning_rate=0.05,
                                           subsample=1.0, random_state=SEED).fit(X[FEATS], e); return self
    def predict(self, X): return np.clip(self.m.predict(X[FEATS]), 0, 1)

def share_matched_cutoffs(h_train, e_train, cuts=CUTS):
    """M1: Hinglish cut-off that sends the same share to the strong model as English does at the Arena cut-off."""
    return {k: float(np.quantile(h_train, 1 - np.mean(e_train >= t))) for k, t in cuts.items()}

# ---------- measures ----------
def sd_ratio(e, c): return float(np.std(e, ddof=1) / np.std(c, ddof=1))
def mae(e, c): return float(np.mean(np.abs(np.asarray(e) - np.asarray(c))))
def orth_slope(x, y):
    sxx, syy, sxy = np.var(x, ddof=1), np.var(y, ddof=1), np.cov(x, y)[0, 1]
    return float((syy - sxx + np.sqrt((syy - sxx) ** 2 + 4 * sxy ** 2)) / (2 * sxy))

def holm(p):
    p = np.asarray(p); order = np.argsort(p); m = len(p); adj = np.empty(m); run = 0
    for rank, i in enumerate(order):
        run = max(run, (m - rank) * p[i]); adj[i] = min(1, run)
    return adj

def routing_table(e, c, cutoff_for_c=None):
    """English decision uses the Arena cut-off. Corrected-Hinglish decision uses the same cut-off,
    or its own (M1) if cutoff_for_c is a dict of cut-offs."""
    rows = []
    for k, t in CUTS.items():
        de = np.asarray(e) >= t
        dc = np.asarray(c) >= (cutoff_for_c[k] if cutoff_for_c else t)
        b, cc = int(np.sum(de & ~dc)), int(np.sum(~de & dc))                  # English-only, Hinglish-only
        p = 1.0 if b + cc == 0 else stats.binomtest(min(b, cc), b + cc, 0.5).pvalue
        kappa = cohen_kappa_score(de, dc) if len(set(de)) > 1 and len(set(dc)) > 1 else float('nan')
        rows.append(dict(cut=k, en_strong=de.mean(), c_strong=dc.mean(), en_only=b, hi_only=cc,
                         agree=float(np.mean(de == dc)), kappa=kappa, p=p))
    t = pd.DataFrame(rows); t['p_holm'] = holm(t['p'].to_numpy()); return t

def auc_hard_easy(score, difficulty, easy=('Easy',), hard=('Hard', 'Very Hard')):
    d = np.asarray(difficulty); m = np.isin(d, easy + hard)
    y = np.isin(d[m], hard).astype(int)
    if len(set(y)) < 2: return float('nan')            # needs both Easy and Hard rows
    return float(roc_auc_score(y, np.asarray(score)[m]))

def bootstrap_ci(fn, n, B=10000, seed=SEED):
    rng = np.random.default_rng(seed); v = [fn(rng.integers(0, n, n)) for _ in range(B)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

# ---------- cross-validation (split by pair; one row per pair so twins never separate) ----------
def cv_predict(make_model, X, e, k=10):
    oof = np.zeros(len(e))
    for tr, te in KFold(k, shuffle=True, random_state=SEED).split(X):
        oof[te] = make_model().fit(X.iloc[tr], e[tr]).predict(X.iloc[te])
    return oof

def cv_share_matched(X, e, k=10):
    """Out-of-fold M1 decisions: one boolean array per cut-off."""
    out = {c: np.zeros(len(e), bool) for c in CUTS}
    for tr, te in KFold(k, shuffle=True, random_state=SEED).split(X):
        cut = share_matched_cutoffs(X['hi_score'].to_numpy()[tr], e[tr])
        for c in CUTS: out[c][te] = X['hi_score'].to_numpy()[te] >= cut[c]
    return out

def mean_agree(e, c, cutoff_for_c=None): return float(routing_table(e, c, cutoff_for_c)['agree'].mean())

def mae_se(e, pred):
    a = np.abs(e - pred); return float(a.std(ddof=1) / np.sqrt(len(a)))

def tune_and_select(X, e, k=10):
    """Plan section 5: tune on CV MAE (ties -> simpler), then apply the selection rule for the final model."""
    cands = {'M2': [lambda K=K: M2(K) for K in (11, 21, 51, 101)],
             'M3': [lambda: M3()],
             'M4': [lambda d=d, n=n: M4(d, n) for d in (1, 2, 3) for n in (25, 50, 100)]}
    best = {}
    for fam, makers in cands.items():
        res = []
        for mk in makers:
            oof = cv_predict(mk, X, e, k); res.append((mae(e, oof), mae_se(e, oof), mk, oof))
        res.sort(key=lambda r: r[0]); best[fam] = res[0]          # makers are listed simplest-first, sort is stable
    m1 = cv_share_matched(X, e, k)
    def agree_dec(dec): return float(np.mean([np.mean((e >= t) == dec[c]) for c, t in CUTS.items()]))
    agree_m1 = agree_dec(m1)
    agree = {f: mean_agree(e, best[f][3]) for f in best}
    final = 'M2'
    for fam in ('M3', 'M4'):
        if best['M2'][0] - best[fam][0] > best['M2'][1] and agree[fam] > agree_m1: final = fam
    return best, agree, agree_m1, final

## B. Final-run functions (unchanged)

In [ ]:
import os, platform, sklearn, scipy
def sha256_lf(path):
    """SHA-256 of the file with Windows line endings (CRLF) turned into LF, so a Windows copy still verifies."""
    b = open(path, 'rb').read().replace(b'\r\n', b'\n')
    return hashlib.sha256(b).hexdigest()

def _refuse_overwrite(path):
    if os.path.exists(path): raise FileExistsError(f'{path} exists. Never overwrite: save a new version name.')

def load_pairs(scores_path, feats_path, difficulty_col=None):
    s = pd.read_csv(scores_path).rename(columns={'english_bert': 'english_score', 'hinglish_bert': 'hinglish_score'})
    f = pd.read_csv(feats_path)
    d = s.merge(f, on='pair_id', validate='one_to_one')
    assert d.pair_id.is_unique and d[['english_score', 'hinglish_score']].notna().all().all(), 'bad ids or missing scores'
    assert d[['english_score', 'hinglish_score']].apply(lambda c: c.between(0, 1)).all().all(), 'score outside 0..1'
    X = pd.DataFrame({'pair_id': d.pair_id, 'hi_score': d.hinglish_score, 'hi_tokens': d.hinglish_tokens,
                      'hi_words': d.hinglish_words, 'hi_fertility': d.hinglish_fertility}).reset_index(drop=True)
    return X, d.english_score.to_numpy(), (d[difficulty_col].to_numpy() if difficulty_col else None)

def paired_agree_vectors(e, c, cutoff_for_c=None):
    """Per-pair agreement with the English decision, averaged over the five cut-offs (for the B1 paired bootstrap)."""
    v = np.zeros(len(e))
    for k, t in CUTS.items():
        v += ((np.asarray(e) >= t) == (np.asarray(c) >= (cutoff_for_c[k] if cutoff_for_c else t)))
    return v / len(CUTS)

def run_final(third, test, expected, plan_sha256, out_dir='.', B=10000):
    """third/test = (scores_path, features_path). expected = dict of the four frozen SHA-256 values.
    Fits on third, applies ONCE to test. Refuses to run if any file is not the frozen one or any output exists."""
    got = {'third_scores': sha256_lf(third[0]), 'third_features': sha256_lf(third[1]),
           'test_scores': sha256_lf(test[0]), 'test_features': sha256_lf(test[1])}
    for k in got: assert got[k] == expected[k], f'{k}: file is NOT the frozen one ({got[k]}). Stop.'
    for name in ('config_corr_v1.json', 'corrected_scores_conf_v1.csv', 'correction_results_v1.csv',
                 'correction_routing_original_v1.csv', 'correction_routing_corrected_v1.csv', 'correction_routing_m1_v1.csv'):
        _refuse_overwrite(os.path.join(out_dir, name))
    Xt, et, _ = load_pairs(*third)
    Xtf = Xt.drop(columns='pair_id')
    best, agree, agree_m1, final = tune_and_select(Xtf, et)
    maker = best[final][2]; model = maker().fit(Xtf, et)
    cfg = dict(plan_sha256=plan_sha256, final_model=model.name, final_family=final,
               cv_mae={f: best[f][0] for f in best}, cv_mae_se={f: best[f][1] for f in best},
               cv_mean_agree={**agree, 'M1': agree_m1}, frozen_input_hashes=got, seed=SEED, cuts=CUTS, folds=10, bootstrap=B,
               versions=dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__,
                             sklearn=sklearn.__version__, scipy=scipy.__version__))
    json.dump(cfg, open(os.path.join(out_dir, 'config_corr_v1.json'), 'w'), indent=2)
    # ---------------- the single look at the test set ----------------
    Xe, ee, diff = load_pairs(*test, difficulty_col='difficulty')
    corr = model.predict(Xe.drop(columns='pair_id'))
    pd.DataFrame({'pair_id': Xe.pair_id, 'english_score': ee, 'hinglish_score': Xe.hi_score,
                  'hinglish_corrected': corr}).to_csv(os.path.join(out_dir, 'corrected_scores_conf_v1.csv'), index=False, lineterminator='\n')
    h = Xe.hi_score.to_numpy(); n = len(ee); res = {}
    res['final model'] = model.name
    res['C1 SD ratio original'] = sd_ratio(ee, h)
    res['C1 SD ratio corrected'] = sd_ratio(ee, corr)
    res['C1 CI'] = bootstrap_ci(lambda i: sd_ratio(ee[i], corr[i]), n, B)
    res['orth slope original / corrected'] = (orth_slope(ee, h), orth_slope(ee, corr))
    res['MAE original / corrected'] = (mae(ee, h), mae(ee, corr))
    res['MAE improvement CI'] = bootstrap_ci(lambda i: mae(ee[i], h[i]) - mae(ee[i], corr[i]), n, B)
    tab0, tab1 = routing_table(ee, h), routing_table(ee, corr)
    res['C2 any Holm p < 0.05 after correction'] = bool((tab1.p_holm < 0.05).any())
    a0 = auc_hard_easy(h, diff, ('Easy',), ('Hard',)); a1 = auc_hard_easy(corr, diff, ('Easy',), ('Hard',))
    res['C3 AUC original / corrected'] = (a0, a1)
    res['C3 AUC change CI'] = bootstrap_ci(lambda i: auc_hard_easy(corr[i], diff[i], ('Easy',), ('Hard',)) - auc_hard_easy(h[i], diff[i], ('Easy',), ('Hard',)), n, B)
    cuts_m1 = share_matched_cutoffs(Xt.hi_score.to_numpy(), et)
    tab_m1 = routing_table(ee, h, cuts_m1)
    if final != 'M2':      # B1 applies only when a learned model is final
        av, am = paired_agree_vectors(ee, corr), paired_agree_vectors(ee, h, cuts_m1)
        res['B1 mean agreement final / M1'] = (float(av.mean()), float(am.mean()))
        res['B1 difference CI (paired bootstrap)'] = bootstrap_ci(lambda i: av[i].mean() - am[i].mean(), n, B)
    tab0.to_csv(os.path.join(out_dir, 'correction_routing_original_v1.csv'), index=False)
    tab1.to_csv(os.path.join(out_dir, 'correction_routing_corrected_v1.csv'), index=False)
    tab_m1.to_csv(os.path.join(out_dir, 'correction_routing_m1_v1.csv'), index=False)
    pd.Series({k: str(v) for k, v in res.items()}).to_csv(os.path.join(out_dir, 'correction_results_v1.csv'), header=['value'])
    return res, tab0, tab1, tab_m1

def verdict(res):
    """The plan's section 7 success criteria, applied mechanically."""
    lo, hi = res['C1 CI']; c1 = lo <= 1 <= hi
    c2 = not res['C2 any Holm p < 0.05 after correction']
    a0, a1 = res['C3 AUC original / corrected']; c3 = (a1 - a0) >= -0.02
    out = {'C1 SD-ratio interval contains 1': c1, 'C2 no cut-off has Holm p < 0.05': c2, 'C3 AUC not lower by more than 0.02': c3}
    if 'B1 difference CI (paired bootstrap)' in res:
        blo, bhi = res['B1 difference CI (paired bootstrap)']; out['B1 beats share-matched M1 (interval excludes 0, above)'] = blo > 0
    out['WORKS (C1 and C2 and C3)'] = c1 and c2 and c3
    return out

## C. Frozen-file check

In [ ]:
EXPECTED = {
    'scores_thirdset_v1.csv':       'dd0bff026a0c87a26cd0c21b57bbcd92a07d5a6f4fc116719e51a0a14f61c6c4',
    'features_thirdset_v1.csv':     '96d29da731349af35caea7015e783f869c78a1e85ba51ea66b2ecdeb6dc1f682',
    'scores_confirmatory_v1.csv':   '02adcb60eb09532bf66fca4997fb510277543e00ad081c7f7e366299a092bb1e',
    'features_confirmatory_v1.csv': '5de6ef62f75a2b95cf8467c27dfe5747d7310450514368f6910befb17d538aa0',
    'corrected_scores_conf_v1.csv': '021a0b8cbf3520fc5097a37d9c7818027e72c22e488873e0e5d03eb5b0233004',
}
for p, h in EXPECTED.items():
    ok = sha256_lf(p) == h; print('OK  ' if ok else 'FAIL', p); assert ok, 'Not the frozen file. Stop.'

## D. Run both exploratory checks

In [ ]:
THIRD=("scores_thirdset_v1.csv","features_thirdset_v1.csv")
TEST=("scores_confirmatory_v1.csv","features_confirmatory_v1.csv")
os.makedirs("explo_out",exist_ok=False)
Xt,et,_=load_pairs(*THIRD); s3=pd.read_csv(THIRD[0]).set_index("pair_id").loc[Xt.pair_id]
Xe,ee,diff=load_pairs(*TEST,difficulty_col="difficulty"); s2=pd.read_csv(TEST[0]).set_index("pair_id").loc[Xe.pair_id]
task=s2.task_type.to_numpy(); h=Xe.hi_score.to_numpy()
corr=pd.read_csv("corrected_scores_conf_v1.csv").set_index("pair_id").loc[Xe.pair_id].hinglish_corrected.to_numpy()
m1=share_matched_cutoffs(Xt.hi_score.to_numpy(),et)

print("=== CHECK 1: by task type and difficulty (no refit; final M4 corrected scores) ===")
rows=[]
def grp(name,k):
    e,o,c=ee[k],h[k],corr[k]
    rows.append(dict(group=name,n=int(k.sum()),sd_ratio_orig=sd_ratio(e,o),sd_ratio_corr=sd_ratio(e,c),
      mae_orig=mae(e,o),mae_corr=mae(e,c),agree_orig=paired_agree_vectors(e,o).mean(),
      agree_M4=paired_agree_vectors(e,c).mean(),agree_M1=paired_agree_vectors(e,o,m1).mean()))
for t in sorted(set(task)): grp("task:"+t,task==t)
for d in ("Easy","Hard"): grp("difficulty:"+d,diff==d)
r1=pd.DataFrame(rows); print(r1.round(3).to_string(index=False))
r1.round(4).to_csv("explo_out/exploratory_by_group_v1.csv",index=False,lineterminator="\n")

print("\n=== CHECK 2: refit on third set WITHOUT Coding, rerun section 7 ===")
keep=(s3.task_type.to_numpy()!="Coding"); print("third-set pairs kept:",int(keep.sum()),"of",len(keep))
Xk=Xt[keep].reset_index(drop=True); ek=et[keep]; Xkf=Xk.drop(columns="pair_id")
best,agree,agree_m1,final=tune_and_select(Xkf,ek)
for f,(m,se,mk,_) in best.items(): print(f"  {f}: {mk().name:24s} CV MAE {m:.4f}+/-{se:.4f} agree {agree[f]:.3f}")
print("  M1 agreement",round(agree_m1,3),"| FINAL by plan rule:",final)
model=best[final][2]().fit(Xkf,ek); c2=model.predict(Xe.drop(columns="pair_id")); n=len(ee); B=10000
lo,hi=bootstrap_ci(lambda i: sd_ratio(ee[i],c2[i]),n,B)
t1=routing_table(ee,c2); a0=auc_hard_easy(h,diff,("Easy",),("Hard",)); a1=auc_hard_easy(c2,diff,("Easy",),("Hard",))
print(f"  C1 SD ratio {sd_ratio(ee,c2):.3f} CI ({lo:.3f},{hi:.3f}) -> contains 1: {lo<=1<=hi}")
print(f"  C2 any Holm<0.05: {bool((t1.p_holm<0.05).any())}  (min Holm p {t1.p_holm.min():.3f})")
print(f"  C3 AUC {a0:.3f} -> {a1:.3f}  (change {a1-a0:+.3f}) -> ok: {(a1-a0)>=-0.02}")
print(f"  MAE {mae(ee,h):.4f} -> {mae(ee,c2):.4f} | orth slope {orth_slope(ee,h):.3f} -> {orth_slope(ee,c2):.3f}")
res2=dict(final=model.name,sd_ratio=sd_ratio(ee,c2),sd_ci=(lo,hi),mae=mae(ee,c2),auc=a1)
if final!="M2":
    m1k=share_matched_cutoffs(Xk.hi_score.to_numpy(),ek)
    av,am=paired_agree_vectors(ee,c2),paired_agree_vectors(ee,h,m1k)
    bl,bh=bootstrap_ci(lambda i: av[i].mean()-am[i].mean(),n,B)
    print(f"  B1 agreement final {av.mean():.3f} vs M1 {am.mean():.3f}, diff CI ({bl:.3f},{bh:.3f}) -> beats M1: {bl>0}")
    res2.update(agree_final=float(av.mean()),agree_m1=float(am.mean()),b1_ci=(bl,bh))
else:
    m1k=share_matched_cutoffs(Xk.hi_score.to_numpy(),ek); print("  B1 not applicable (M2 final)")
    res2.update(agree_final=float(paired_agree_vectors(ee,c2).mean()),agree_m1=float(paired_agree_vectors(ee,h,m1k).mean()))
print(t1.round(3).to_string(index=False))
pd.Series({k:str(v) for k,v in res2.items()}).to_csv("explo_out/exploratory_no_coding_v1.csv",header=["value"],lineterminator="\n")


## E. Download

In [ ]:
import shutil
shutil.make_archive('explo_out_v1', 'zip', 'explo_out')
files.download('explo_out_v1.zip')